# AIS YOLO TaskBoard Pose Train

Train a YOLO pose model for one `task_board` bbox and four fixed board-local keypoints.

Assumptions:
- YOLO pose label format: `class x_center y_center width height kpt0_x kpt0_y ... kpt3_x kpt3_y`
- keypoint order is fixed in `task_board_base_link`, not image-order corners.
- horizontal/vertical flips are disabled because fixed local keypoint identity must be preserved.

In [1]:
from pathlib import Path
import random

import yaml


def find_src_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pixi.toml").exists() and (path / "ais").exists():
            return path
        nested_src = path / "ws_aic" / "src"
        if (nested_src / "pixi.toml").exists() and (nested_src / "ais").exists():
            return nested_src
    raise RuntimeError("Could not find ws_aic/src root. Run this notebook under ws_aic/src.")


SRC_ROOT = find_src_root(Path.cwd().resolve())
WS_ROOT = SRC_ROOT.parent

TARGET = "TASK_BOARD"
DATASET_DIR = WS_ROOT / "data" / "yolo" / "approach" / TARGET
DATA_YAML = DATASET_DIR / "data.yaml"

KEYPOINT_DIMS = 2
KEYPOINT_NAMES = [
    "board_neg_x_pos_y",
    "board_pos_x_pos_y",
    "board_pos_x_neg_y",
    "board_neg_x_neg_y",
]
CLASS_NAMES = {0: "task_board"}
IMAGE_EXTS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DATA_YAML: {DATA_YAML}")

SRC_ROOT: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/src
DATASET_DIR: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD
DATA_YAML: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/data.yaml


## Dataset yaml

Create or refresh the Ultralytics pose dataset config after labels are collected.

In [2]:
dataset_cfg = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "kpt_shape": [len(KEYPOINT_NAMES), KEYPOINT_DIMS],
    "flip_idx": list(range(len(KEYPOINT_NAMES))),
    "names": CLASS_NAMES,
    "kpt_names": {0: KEYPOINT_NAMES},
}

DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text(yaml.safe_dump(dataset_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

path: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD
train: images/train
val: images/val
kpt_shape:
- 4
- 2
flip_idx:
- 0
- 1
- 2
- 3
names:
  0: task_board
kpt_names:
  0:
  - board_neg_x_pos_y
  - board_pos_x_pos_y
  - board_pos_x_neg_y
  - board_neg_x_neg_y



## Label check

Check image/label counts, expected column count, and normalized coordinate ranges before training.

In [3]:
def resolve_split_dir(cfg: dict, split: str) -> Path:
    split_path = Path(cfg[split]).expanduser()
    if split_path.is_absolute():
        return split_path
    root = Path(cfg.get("path", DATA_YAML.parent)).expanduser()
    if not root.is_absolute():
        root = DATA_YAML.parent / root
    return (root / split_path).resolve()


def label_path_for_image(image_path: Path) -> Path:
    parts = list(image_path.parts)
    if "images" in parts:
        parts[parts.index("images")] = "labels"
        return Path(*parts).with_suffix(".txt")
    return image_path.with_suffix(".txt")


def iter_images(image_dir: Path) -> list[Path]:
    if not image_dir.exists():
        return []
    return sorted(p for p in image_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def validate_label_file(label_path: Path, expected_cols: int) -> list[str]:
    issues = []
    if not label_path.exists():
        return [f"missing label: {label_path}"]
    lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not lines:
        return [f"empty label: {label_path}"]
    for line_idx, line in enumerate(lines, start=1):
        try:
            values = [float(v) for v in line.split()]
        except ValueError:
            issues.append(f"non-numeric label values: {label_path}:{line_idx}")
            continue
        if len(values) != expected_cols:
            issues.append(f"wrong column count {len(values)} != {expected_cols}: {label_path}:{line_idx}")
            continue
        coords = values[1:]
        if any(v < 0.0 or v > 1.0 for v in coords):
            issues.append(f"normalized coordinate out of range: {label_path}:{line_idx}")
    return issues


cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
expected_cols = 5 + len(KEYPOINT_NAMES) * KEYPOINT_DIMS
all_issues = []
for split in ["train", "val"]:
    image_dir = resolve_split_dir(cfg, split)
    images = iter_images(image_dir)
    print(f"{split}: images={len(images)} dir={image_dir}")
    for image_path in images:
        all_issues.extend(validate_label_file(label_path_for_image(image_path), expected_cols))

print(f"expected label columns: {expected_cols}")
print(f"issues: {len(all_issues)}")
for issue in all_issues[:30]:
    print(issue)
if all_issues:
    raise RuntimeError("Fix label issues before training.")

train: images=144 dir=/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/images/train
val: images=74 dir=/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/images/val
expected label columns: 13
issues: 0


## Train

`MODEL_NAME` must be a pose model. The local `yolo11s-pose.pt` is used when present.

In [4]:
from ultralytics import YOLO


LOCAL_MODEL = SRC_ROOT / "ais" / "ais_yolo_train" / "yolo11s-pose.pt"
MODEL_NAME = str(LOCAL_MODEL) if LOCAL_MODEL.exists() else "yolo11s-pose.pt"
OUTPUT_DIR = WS_ROOT / "model" / "ais_yolo" / "approach" / TARGET

EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = 0  # CPU: "cpu"
WORKERS = 8
PATIENCE = 20

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(OUTPUT_DIR.parent),
    name=OUTPUT_DIR.name,
    patience=PATIENCE,
    save=True,
    save_period=10,
    plots=True,
    device=DEVICE,
    workers=WORKERS,
    fliplr=0.0,
    flipud=0.0,
)

best_pt = OUTPUT_DIR / "weights" / "best.pt"
print(f"best.pt: {best_pt}")

New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.40 🚀 Python-3.10.0 torch-2.10.0+cu130 CUDA:0 (NVIDIA GB10, 122566MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-pose.pt

/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


YOLO11s-pose summary: 197 layers, 9,715,051 parameters, 9,715,035 gradients, 22.5 GFLOPs

Transferred 505/541 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3526.9±1076.4 MB/s, size: 96.1 KB)
train: Scanning /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/labels/train... 144 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 144/144 2.2Kit/s 0.1s
train: New cache created: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/labels/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1834.8±1686.7 MB/s, size: 84.3 KB)
val: Scanning /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/labels/val... 74 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 1.8Kit/s 0.0s
val: New cache created: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/labels/val.cache
opt

## Resume or validate only

In [ ]:
# Resume full training from last.pt.
# resume_pt = OUTPUT_DIR / "weights" / "last.pt"
# model = YOLO(str(resume_pt))
# results = model.train(resume=True)

In [ ]:
# Validate without training.
# model = YOLO(str(OUTPUT_DIR / "weights" / "best.pt"))
# metrics = model.val(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, device=DEVICE)
# print(metrics)

## Sample prediction

In [5]:
import cv2
import matplotlib.pyplot as plt

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
sample_images = iter_images(resolve_split_dir(cfg, "val")) or iter_images(resolve_split_dir(cfg, "train"))
if not sample_images:
    raise RuntimeError(f"No sample images found under {DATASET_DIR}")

sample = random.choice(sample_images)
model = YOLO(str(OUTPUT_DIR / "weights" / "best.pt"))
pred = model.predict(
    source=str(sample),
    imgsz=IMGSZ,
    device=DEVICE,
    save=True,
    project=str(OUTPUT_DIR.parent),
    name=f"{OUTPUT_DIR.name}_predict",
)
annotated = pred[0].plot()
plt.figure(figsize=(8, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(sample.name)
print(sample)


image 1/1 /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/images/val/ep00378_right.jpg: 576x640 1 task_board, 17.7ms
Speed: 1.2ms preprocess, 17.7ms inference, 1.5ms postprocess per image at shape (1, 3, 576, 640)
Results saved to /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/model/ais_yolo/approach/TASK_BOARD_predict
/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/TASK_BOARD/images/val/ep00378_right.jpg
